# vi-function-calling-slm · 02 SFT (QLoRA) Qwen3-1.7B

Cần **GPU T4**. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Unsloth cần GPU có CUDA capability ≥ 7.0: T4 (7.5) chạy được, **P100 (6.0) thì không**.
Input: code + output của notebook 01 (`data/gen/*.jsonl`). Kaggle: add dataset `vi-function-calling-slm-code`
và output notebook 01 làm input; Colab: chạy notebook 01 trước, dữ liệu nằm sẵn trên Drive.
Secret: `HF_TOKEN` (đẩy adapter lên `DuongCodeAI/vi-fc-qwen3-1.7b`).
Trên Colab trainer lưu checkpoint vào `outputs/sft` trên Drive, bị ngắt thì chạy lại notebook là train tiếp.

- Model: `unsloth/Qwen3-1.7B` (Apache-2.0), load 4-bit, LoRA r=16 trên mọi lớp linear
- Định dạng: chat template gốc của Qwen3 với `tools=`, `enable_thinking=False`
- Chỉ tính loss trên lượt assistant (`train_on_responses_only`)
- 2 epoch, eval trên val

In [ ]:
import glob, json, os, sys

IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/vi_fc/__init__.py", recursive=True)[0].rsplit("/vi_fc", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vi-function-calling-slm" if IN_COLAB else os.path.abspath("vi-function-calling-slm")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vi-function-calling-slm {REPO}
    else:  # đã clone ở lần chạy trước -> lấy bản code mới nhất
        !git -C {REPO} pull -q
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vi-function-calling-slm" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in (["/kaggle/input"] if IN_KAGGLE else []) + [WORK, REPO]:
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


DATA = os.path.dirname(find("gen/train.jsonl", "chạy notebook 01 trước"))
print(CODE, DATA)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q unsloth

token = secret("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token
else:
    print("thiếu HF_TOKEN, sẽ không push")

In [ ]:
# import unsloth trước transformers/trl để nó patch kịp
from unsloth import FastLanguageModel
import torch

MAX_LEN = 3072
model, tokenizer = FastLanguageModel.from_pretrained("unsloth/Qwen3-1.7B", max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=0,
)

## Dữ liệu -> text theo chat template
Xem độ dài: phần system + schema 17 tool chiếm phần lớn, ảnh hưởng thẳng tới độ trễ prefill trên CPU.

In [ ]:
from datasets import Dataset
from vi_fc.prompt import build_messages, record_messages
from vi_fc.tools import openai_tools

TOOLS = openai_tools()


def load(name):
    return [json.loads(l) for l in open(f"{DATA}/{name}.jsonl", encoding="utf-8")]


def to_text(rec):
    return tokenizer.apply_chat_template(record_messages(rec), tools=TOOLS, tokenize=False, enable_thinking=False)


train, val = load("train"), load("val")
ds_train = Dataset.from_list([{"text": to_text(r)} for r in train]).shuffle(seed=0)
ds_val = Dataset.from_list([{"text": to_text(r)} for r in val])

prefix = tokenizer.apply_chat_template(build_messages("x", None), tools=TOOLS, tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False)
lens = sorted(len(tokenizer(t).input_ids) for t in ds_train["text"])
print(f"{len(train)} train / {len(val)} val")
print("token prompt cố định (system + tools):", len(tokenizer(prefix).input_ids))
print(f"token/ví dụ p50={lens[len(lens)//2]} max={lens[-1]}")
print(ds_train[0]["text"][-700:])

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

bf16 = torch.cuda.is_bf16_supported()
args = SFTConfig(
    output_dir="outputs/sft", dataset_text_field="text", packing=False,
    per_device_train_batch_size=4, gradient_accumulation_steps=4, num_train_epochs=2,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03, weight_decay=0.01,
    optim="adamw_8bit", fp16=not bf16, bf16=bf16, logging_steps=10,
    eval_strategy="steps", eval_steps=100, per_device_eval_batch_size=8,
    # Colab có thể ngắt giữa chừng -> lưu checkpoint lên Drive để train tiếp; Kaggle chạy một mạch nên không lưu
    save_strategy="steps" if IN_COLAB else "no", save_steps=100, save_total_limit=1,
    report_to="none", seed=0,
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds_train, eval_dataset=ds_val, args=args)
# lượt user (kể cả state <xe>) và system/tools không tính loss, chỉ học phần assistant trả lời
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")

# kiểm tra mask: phần còn lại sau khi bỏ -100 phải chỉ là câu trả lời của assistant
ex = trainer.train_dataset[0]
print(repr(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])))

In [ ]:
stats = trainer.train(resume_from_checkpoint=bool(glob.glob("outputs/sft/checkpoint-*")))
print(stats.metrics)

In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label="train"); plt.plot(*zip(*ev), "o-", label="val")
plt.yscale("log"); plt.legend(); plt.xlabel("step"); plt.title("SFT loss")
plt.savefig("sft_loss.png", dpi=110, bbox_inches="tight")

## Đánh giá nhanh trên val (200 câu) và bộ viết tay

In [ ]:
from vi_fc.evaluate import compute_metrics, load_jsonl, predict
from vi_fc.inference import FunctionCaller

FastLanguageModel.for_inference(model)
fc = FunctionCaller("hf", model, tokenizer=tokenizer, guard=False)
manual = load_jsonl(find("data/test_manual.jsonl", "thiếu data/test_manual.jsonl của repo"))
for name, recs in (("val", val[:200]), ("manual", manual)):
    rows = predict(recs, fc, "sft", name, f"sft_{name}.jsonl")
    m = compute_metrics(rows)
    print(name, json.dumps({k: v for k, v in m.items() if k != "exact_by_scenario"}, ensure_ascii=False))
    print("  theo kịch bản:", m["exact_by_scenario"])

In [ ]:
# soi các câu sai trên bộ viết tay
from vi_fc.match import actions_match
from vi_fc.parse import Action

for r in load_jsonl("sft_manual.jsonl"):
    if not actions_match(Action.from_dict(r["gold"]), Action.from_dict(r["pred"])):
        print(r["user"]); print("   gold:", r["gold"]); print("   pred:", r["raw"][:200])

## Lưu adapter + đẩy lên Hub

In [ ]:
model.save_pretrained("vi-fc-sft-lora")
tokenizer.save_pretrained("vi-fc-sft-lora")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi

    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo("DuongCodeAI/vi-fc-qwen3-1.7b", exist_ok=True)
    api.upload_folder(folder_path="vi-fc-sft-lora", repo_id="DuongCodeAI/vi-fc-qwen3-1.7b", path_in_repo="sft-lora")